In [2]:
### Box Encoding & Decoding

import torch

'''
1. 앵커 박스와 바운딩 박스에 관하여:
두 박스 모두 [N, 4] 형태의 2차원 PyTorch 텐서!
N은 박스 개수, 4는 좌표값 4개
→ 아래 코드에서는 네 좌표값을,
중심점 좌표와 크기([x_center, y_center, width, height])
형태로 정의!

e.g.1. anchors[:, 0]은 앵커 박스 중심 x 좌표
e.g.2. anchors[:, 2]는 앵커 박스 가로 길이(Width)

2. 위치 오프셋과 크기 오프셋에 관하여:
  위치 오프셋 → 앵커 크기 대비 실제 박스가 중심점으로부터 떨어진 비율 계산
  크기 오프셋 → 앵커 크기 대비 실제 박스 크기 비율에 로그를 취한 스케일 오프셋
'''

# 앵커 박스에 대한 실제 바운딩 박스의 오프셋 계산
def encode_boxes(anchors, gt_boxes):
  tx = (gt_boxes[:, 0] - anchors[:, 0]) / (anchors[:, 2] + 1e-4) # (분모 ≠ 0)
  ty = (gt_boxes[:, 1] - anchors[:, 1]) / (anchors[:, 3] + 1e-4)
  tw = torch.log(gt_boxes[:, 2] / anchors[:, 2])
  th = torch.log(gt_boxes[:, 3] / anchors[:, 3])
  return torch.stack([tx, ty, tw, th], dim=1)

def decode_boxes(anchors, offsets):
  # 예측된 오프셋을 사용하여 앵커 박스 반환
  x = offsets[:, 0] * anchors[:, 2] + anchors[:, 0]
  y = offsets[:, 1] * anchors[:, 3] + anchors[:, 1]
  w = torch.exp(offsets[:, 2]) * anchors[:, 2]
  h = torch.exp(offsets[:, 3]) * anchors[:, 3]
  return torch.stack([x, y, w, h], dim=1)